# NB11 · Capstone Starter and Evaluation Harness

Pairs with **Module 8**.

This notebook gives you the scaffolding so you spend your time on the science rather than the plumbing. It is **deliberately incomplete**: the feature map, the model and the analysis are yours.

What it does provide is the parts that are easy to get *silently* wrong — fold hygiene, leakage-free preprocessing, matched baselines, ablation checks, a noise sweep and resource counting.

The demo below runs the whole protocol on a small built-in dataset so you can see the shape of the output. Swap in your own data at section 1.

## 0. Setup — one seed, used everywhere

In [1]:
import warnings, time, numpy as np, pandas as pd
warnings.filterwarnings("ignore")

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import (zz_feature_map, pauli_feature_map,
                                    efficient_su2, unitary_overlap)
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, balanced_accuracy_score
from scipy import stats

SEED  = 0                     # ONE seed constant. Use it everywhere.
N_FOLDS = 5
M_DEMO  = 120                 # demo sample size -- see the sizing note in sec. 4
rng = np.random.default_rng(SEED)

ONE_Q = ["rx", "ry", "rz", "h", "x", "y", "z", "sx", "u"]
TWO_Q = ["cx", "cz"]
def to_basis(c):
    return transpile(c, basis_gates=ONE_Q + TWO_Q, optimization_level=0)

# Everything in this harness is sampled on AerSimulator. No statevectors: a
# capstone that reports numbers an exact simulator produced has not measured
# what a device would give you.
SHOTS_KERNEL = 2048
sim_dev = AerSimulator(max_parallel_experiments=0, max_parallel_threads=0)

import qiskit
print("qiskit", qiskit.__version__, "| seed", SEED)

qiskit 2.5.1 | seed 0


---

## 1. Your data goes here

Replace `load_data()` with your own loader. It must return `X` of shape `(n_samples, n_features)` and `y` of shape `(n_samples,)` with labels in $\{-1, +1\}$.

Print the shape and the class balance. An imbalanced set changes which metric you should report.

In [2]:
def load_data():
    """REPLACE ME with your dataset.

    The default is a small synthetic set with genuine feature interactions --
    deliberately chosen so a linear model does NOT solve it, which is the
    property you want in a capstone dataset.
    """
    g = np.random.default_rng(SEED)
    n, d = M_DEMO, 6
    X = g.uniform(-1, 1, (n, d))
    signal = np.sin(2.5 * X[:, 0] * X[:, 1]) + 0.6 * np.cos(3 * X[:, 2])
    y = np.where(signal > np.median(signal), 1, -1)
    X += g.normal(0, 0.05, X.shape)
    return X, y

X_all, y = load_data()
print("X", X_all.shape, " y", y.shape)
print("class balance:", dict(zip(*np.unique(y, return_counts=True))))
print("majority-class rate: %.3f" % (max(np.bincount((y + 1) // 2)) / len(y)))

X (120, 6)  y (120,)
class balance: {np.int64(-1): np.int64(60), np.int64(1): np.int64(60)}
majority-class rate: 0.500


### Write your question and your falsifier here, before running anything

> **Question.** *(edit me)* Does a ZZ feature-map kernel beat a tuned RBF SVM on this dataset under stratified 5-fold cross-validation at matched hyperparameter budget?
>
> **Falsifier.** *(edit me)* I will conclude the quantum kernel does **not** help if its CV mean fails to exceed the best classical baseline by more than one pooled standard deviation.

Writing this down first is the cheapest protection against deciding what counts as success after seeing the numbers.

---

## 2. The fold object — created once, reused everywhere

Every table in your report must use these exact folds, or the comparisons are not comparisons.

In [3]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X_all, y))
print(f"{N_FOLDS} folds, test sizes:", [len(te) for _, te in FOLDS])

5 folds, test sizes: [24, 24, 24, 24, 24]


## 3. Leakage-free preprocessing

`make_pipeline` returns a *factory*, not a fitted object, so the harness is forced to fit it inside each fold. Selection, scaling and imputation all belong here.

In [4]:
def select_ttest(Xtr, ytr, k):
    t, _ = stats.ttest_ind(Xtr[ytr == 1], Xtr[ytr == -1], axis=0)
    return np.argsort(-np.abs(np.nan_to_num(t)))[:k]

def select_all(Xtr, ytr, k):
    return np.arange(min(k, Xtr.shape[1]))

def make_preprocessor(n_features=6, bandwidth=1.0, selector=select_all):
    """Returns fit(Xtr, ytr) -> transform(X). Everything learned from TRAIN only."""
    def fit(Xtr, ytr):
        genes  = selector(Xtr, ytr, n_features)
        scaler = MinMaxScaler((0, np.pi)).fit(Xtr[:, genes])
        def transform(X):
            return scaler.transform(X[:, genes]) * bandwidth
        return transform
    return fit

## 4. Models

A quantum kernel and the classical baselines it will be compared against.

**Everything is sampled on `AerSimulator`.** That is the expensive choice and it is deliberate: a sampled kernel costs one circuit per *pair*, so the work is $O(M^2)$ where an exact simulator would be $O(M)$, and it is not guaranteed PSD. Both of those are properties of the real problem, and a capstone that hides them has not measured what it claims to have measured.

That cost is why the demo dataset is `M_DEMO = 120` rather than a few hundred. At $M=120$ with 5 folds one cross-validated row is about 34,000 circuits; at $M=240$ it is four times that. **Size your capstone dataset from this number, not from how much data you have.**

In [5]:
_OV_CACHE = {}

def overlap_circuit(fm):
    """Transpiled, still-parameterised U(p1)^dag U(p2). Cached per feature map."""
    key = id(fm)
    if key not in _OV_CACHE:
        ov = unitary_overlap(fm, fm)
        ov.measure_all()
        _OV_CACHE[key] = transpile(ov, sim_dev)
    return _OV_CACHE[key]

def _run_pairs(tov, pairs, shots, seed):
    if not pairs:
        return np.zeros(0)
    cols = {}
    for prm in tov.parameters:                        # named p1[k] / p2[k]
        k, side = int(prm.name.split("[")[1][:-1]), prm.name[1]
        cols[prm] = [float(ab[0 if side == "1" else 1][k]) for ab in pairs]
    res = sim_dev.run(tov, shots=shots, parameter_binds=[cols],
                      seed_simulator=seed).result()
    return np.array([res.data(j)["counts"].get("0x0", 0) / shots
                     for j in range(len(pairs))])

def quantum_gram(fm, A, B=None, shots=SHOTS_KERNEL, seed=SEED):
    """Sampled |<phi(a)|phi(b)>|^2 -- one circuit per PAIR, as on hardware.

    Symmetric case: upper triangle only, diagonal pinned to 1 (it is exactly 1,
    so sampling it would only add noise to a known value).
    """
    tov = overlap_circuit(fm)
    A = np.asarray(A, float)
    if B is None:
        idx  = [(i, j) for i in range(len(A)) for j in range(i + 1, len(A))]
        vals = _run_pairs(tov, [(A[i], A[j]) for i, j in idx], shots, seed)
        K = np.eye(len(A))
        for (i, j), v in zip(idx, vals):
            K[i, j] = K[j, i] = v
        return K
    B = np.asarray(B, float)
    return _run_pairs(tov, [(a, b) for a in A for b in B],
                      shots, seed).reshape(len(A), len(B))

def product_gram(A, B=None):
    """Closed-form Ry product kernel -- the no-entanglement control. No circuits."""
    B = A if B is None else B
    return np.prod(np.cos((A[:, None, :] - B[None, :, :]) / 2) ** 2, axis=2)

def make_zz_kernel(n, reps=1, entanglement="linear"):
    fm = zz_feature_map(n, reps=reps, entanglement=entanglement)
    return lambda A, B=None: quantum_gram(fm, A, B), fm

def make_z_kernel(n, reps=1):
    """The no-entanglement control. Note you cannot get this by passing
    entanglement=[] to zz_feature_map -- Qiskit 2.x raises. The ZZ terms live
    in the *Pauli list*, so you drop them with paulis=["Z"]."""
    fm = pauli_feature_map(n, reps=reps, paulis=["Z"])
    return lambda A, B=None: quantum_gram(fm, A, B), fm

def mean_offdiag(K):
    return K[~np.eye(len(K), dtype=bool)].mean()

## 5. The cross-validation harness

One function for kernel models, one for ordinary sklearn estimators. Both take the same folds and the same preprocessor factory, so every row of your results table is directly comparable.

In [6]:
def cv_kernel(kernel_fn, prep_factory, C=1.0, folds=FOLDS, metric="accuracy",
              data=None):
    """data=None uses the full set. Pass (X, y) to run on a subsample -- the
    noise section does, because a noisy kernel needs a density-matrix simulation
    for each of the O(M^2) pairs."""
    Xd, yd = (X_all, y) if data is None else data
    scores, diagnostics = [], []
    for itr, ite in folds:
        transform = prep_factory(Xd[itr], yd[itr])
        A, B = transform(Xd[itr]), transform(Xd[ite])
        K, Kt = kernel_fn(A), kernel_fn(B, A)
        clf = SVC(kernel="precomputed", C=C).fit(K, yd[itr])
        if metric == "auc":
            scores.append(roc_auc_score(yd[ite], clf.decision_function(Kt)))
        elif metric == "balanced":
            scores.append(balanced_accuracy_score(yd[ite], clf.predict(Kt)))
        else:
            scores.append(accuracy_score(yd[ite], clf.predict(Kt)))
        diagnostics.append((mean_offdiag(K), np.linalg.eigvalsh(K).min()))
    d = np.array(diagnostics)
    return np.array(scores), dict(mean_K=d[:, 0].mean(), min_eig=d[:, 1].min())

def cv_classical(estimator_factory, prep_factory, folds=FOLDS, metric="accuracy"):
    scores = []
    for itr, ite in folds:
        transform = prep_factory(X_all[itr], y[itr])
        clf = estimator_factory().fit(transform(X_all[itr]), y[itr])
        if metric == "auc":
            scores.append(roc_auc_score(y[ite], clf.decision_function(transform(X_all[ite]))))
        elif metric == "balanced":
            scores.append(balanced_accuracy_score(y[ite], clf.predict(transform(X_all[ite]))))
        else:
            scores.append(accuracy_score(y[ite], clf.predict(transform(X_all[ite]))))
    return np.array(scores)

def row(name, scores, extra=""):
    return f"{name:<34}{scores.mean():>8.3f}{scores.std():>8.3f}   {np.round(scores,2)} {extra}"

---

## 6. The baseline table

Majority class is the row people omit, and it is the floor everything must clear. Tune the classical models as hard as the quantum one.

**Two things to notice in the demo output.** Logistic regression and the linear SVM land *below* the majority-class rate. That is not a bug — the built-in dataset is deliberately nonlinear, and a linear decision boundary fitted to one fold can generalise worse than a constant predictor on the next. It is also why the majority row belongs in the table: without it you would read 0.44 as 'bad' rather than 'worse than guessing'.

And the tuned RBF SVM sits far above everything quantum. In this demo the honest conclusion is that the quantum kernel loses, clearly. Say so. A capstone that reaches that conclusion with a clean protocol scores better than one that reaches the opposite with a dirty one.

In [7]:
N_FEAT, BW = 6, 1.0
prep = make_preprocessor(N_FEAT, BW, select_all)

print(f"{'model':<34}{'mean':>8}{'s.d.':>8}   folds")
print("-" * 76)
print(row("Majority class",
          cv_classical(lambda: DummyClassifier(strategy="most_frequent"), prep)))
print(row("Logistic regression",
          cv_classical(lambda: LogisticRegression(max_iter=5000), prep)))
print(row("SVM (linear)", cv_classical(lambda: SVC(kernel="linear"), prep)))
print(row("SVM (RBF, tuned)",
          cv_classical(lambda: GridSearchCV(SVC(),
                       {"C": [0.1, 1, 10], "gamma": ["scale", 0.1, 1]}, cv=3), prep)))

kfn, fm_used = make_zz_kernel(N_FEAT, reps=1, entanglement="linear")
s, diag = cv_kernel(kfn, prep, C=1.0)
print(row("QSVM  ZZ linear reps=1", s,
          f"meanK={diag['mean_K']:.3f} min_eig={diag['min_eig']:+.1e}"))

model                                 mean    s.d.   folds
----------------------------------------------------------------------------
Majority class                       0.500   0.000   [0.5 0.5 0.5 0.5 0.5] 
Logistic regression                  0.533   0.031   [0.54 0.58 0.5  0.54 0.5 ] 
SVM (linear)                         0.558   0.033   [0.58 0.58 0.5  0.58 0.54] 


SVM (RBF, tuned)                     0.892   0.050   [0.96 0.92 0.83 0.92 0.83] 


QSVM  ZZ linear reps=1               0.675   0.061   [0.75 0.62 0.58 0.71 0.71] meanK=0.017 min_eig=+3.6e-01


**Read the majority-class row first.** If your quantum model ties it, the model is predicting one class for everything and no other number in the table means anything yet.

## 7. Tuning — on inner splits only

Bandwidth is usually the highest-value knob. Tune it here, not on the folds you report.

In [8]:
inner = list(StratifiedKFold(3, shuffle=True, random_state=SEED + 1).split(X_all, y))
print(f"{'bandwidth c':>12}{'inner mean':>12}{'mean K':>10}")
best = (None, -1)
for c in [0.05, 0.1, 0.25, 0.5, 1.0]:
    p = make_preprocessor(N_FEAT, c, select_all)
    s, d = cv_kernel(kfn, p, C=1.0, folds=inner)
    print(f"{c:>12}{s.mean():>12.3f}{d['mean_K']:>10.3f}")
    if s.mean() > best[1]:
        best = (c, s.mean())
print(f"\nbest bandwidth on INNER splits: c = {best[0]}")

 bandwidth c  inner mean    mean K


        0.05       0.667     0.692


         0.1       0.683     0.297


        0.25       0.725     0.031


         0.5       0.792     0.017


         1.0       0.700     0.016

best bandwidth on INNER splits: c = 0.5


---

## 8. The ablation table

The experiment almost nobody runs, and the most informative one available to you. Each row removes one ingredient; the first row that does *not* drop tells you that ingredient was not contributing.

In [9]:
BW_BEST = best[0]
prep_b = make_preprocessor(N_FEAT, BW_BEST, select_all)

print(f"{'configuration':<34}{'mean':>8}{'s.d.':>8}   folds")
print("-" * 76)
s_full, _ = cv_kernel(kfn, prep_b, C=1.0)
print(row("Full quantum model (ZZ linear)", s_full))

kfn_noent, _ = make_z_kernel(N_FEAT, reps=1)      # Z-only: no two-qubit terms
s_ne, _ = cv_kernel(kfn_noent, prep_b, C=1.0)
print(row("Entanglers removed (Z only)", s_ne))

s_prod, _ = cv_kernel(lambda A, B=None: product_gram(A, B), prep_b, C=1.0)
print(row("Product kernel (closed form)", s_prod))

print(row("SVM (RBF, tuned)",
          cv_classical(lambda: GridSearchCV(SVC(),
                       {"C": [0.1, 1, 10], "gamma": ["scale", 0.1, 1]}, cv=3), prep_b)))

perm = rng.permutation(X_all.shape[1])
prep_shuf = make_preprocessor(N_FEAT, BW_BEST,
                              lambda Xtr, ytr, k: perm[:k])
s_shuf, _ = cv_kernel(kfn, prep_shuf, C=1.0)
print(row("Feature-to-qubit order shuffled", s_shuf))

configuration                         mean    s.d.   folds
----------------------------------------------------------------------------


Full quantum model (ZZ linear)       0.700   0.096   [0.83 0.79 0.62 0.67 0.58] 


Entanglers removed (Z only)          0.792   0.070   [0.88 0.79 0.67 0.79 0.83] 
Product kernel (closed form)         0.708   0.087   [0.88 0.67 0.62 0.67 0.71] 


SVM (RBF, tuned)                     0.892   0.042   [0.96 0.88 0.83 0.92 0.88] 


Feature-to-qubit order shuffled      0.550   0.093   [0.58 0.54 0.46 0.71 0.46] 


### Ablation 2: are the ansatz parameters alive?

If you use a parameterised feature map, run this before trusting anything. A fixed unitary applied *after* the encoding cancels exactly in the overlap, so trailing blocks contribute nothing.

In [10]:
def ansatz_is_alive(build_fm, n_params, n=N_FEAT, shots=32768):
    """Sampled version. Same Aer seed for both settings, so identical
    distributions give bit-identical counts -- a sharper test than any
    tolerance, and it does not need an exact simulator."""
    Xt = rng.uniform(0, np.pi/4, (6, n))
    K0 = quantum_gram(build_fm(np.zeros(n_params)), Xt, shots=shots, seed=SEED)
    K1 = quantum_gram(build_fm(rng.uniform(0, 2*np.pi, n_params)), Xt,
                      shots=shots, seed=SEED)
    delta = np.abs(K0 - K1).max()
    noise = 1/np.sqrt(shots)
    alive = delta > 0
    print(f"  max|dK| = {delta:.2e} ({delta/noise:.1f} x shot noise)  ->  "
          f"{'ALIVE' if alive else 'INERT - the ansatz does nothing'}")
    return alive

# Example: encoding THEN ansatz -- the pattern that cancels.
xp = ParameterVector("x", N_FEAT)
_d = QuantumCircuit(N_FEAT)
for q in range(N_FEAT):
    _d.ry(xp[q], q)
_su2 = efficient_su2(N_FEAT, reps=1, entanglement="full", parameter_prefix="theta")
_tp  = sorted(_su2.parameters, key=lambda p: int(p.name.split("[")[1][:-1]))

def build_after(theta):
    fm = _d.compose(_su2)
    return fm.assign_parameters({p: theta[i] for i, p in enumerate(_tp)})

print("ansatz composed AFTER the encoding:")
ansatz_is_alive(build_after, len(_tp))

ansatz composed AFTER the encoding:


  max|dK| = 0.00e+00 (0.0 x shot noise)  ->  INERT - the ansatz does nothing


np.False_

If that prints **INERT**, your parameters are decoration. Interleave instead — $W_1 D(x) W_2 D(x) W_3$ — so no block can be commuted out.

---

## 9. Leakage check

Train on permuted labels. Held-out accuracy must collapse to chance. If it does not, information is crossing the fold boundary and every number above is wrong.

In [11]:
y_true_backup = y.copy()
y = rng.permutation(y_true_backup)
s_shuffled, _ = cv_kernel(kfn, prep_b, C=1.0)
y = y_true_backup

chance = max(np.bincount((y + 1) // 2)) / len(y)
print(f"shuffled-label CV accuracy : {s_shuffled.mean():.3f} +/- {s_shuffled.std():.3f}")
print(f"chance (majority) rate     : {chance:.3f}")
print("PASS - no leakage detected" if s_shuffled.mean() < chance + 2*s_shuffled.std()
      else "FAIL - suspicious, investigate preprocessing")

shuffled-label CV accuracy : 0.483 +/- 0.057
chance (majority) rate     : 0.500
PASS - no leakage detected


---

## 10. Noise sweep and repeats

Report the metric, its spread, **and** the mean off-diagonal $K$ that explains both. With shots enabled, repeat the whole evaluation — noise inflates variance far more than it shifts the mean.

**Sizing warning, and it matters.** Everything above is already $O(M^2)$ circuits. The noise sweep multiplies that by the number of noise levels, and each circuit becomes a *density-matrix* simulation rather than a statevector one — which is roughly a hundred times more expensive per circuit at this width. At $M = 240$ with 5 folds that is about 138,000 density-matrix simulations per noise level: hours, not minutes. I sized this cell wrong the first time and it ran for twenty minutes without finishing.

So the noise section runs on a **stratified subsample** (`M_NOISE = 60`, 3 folds), which is far cheaper and finishes in about a minute. Subsampling here is legitimate; hiding it is not. Put the subsample size and fold count in your write-up next to the numbers. Absolute accuracies in this section are noisier than section 6's and are **not comparable to them** — what is comparable is the trend across `p`, which is the whole point of the sweep.

In [12]:
def depolarizing_model(p):
    if p <= 0:
        return None
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p, 1), ONE_Q)
    nm.add_all_qubit_quantum_error(depolarizing_error(p, 2), TWO_Q)
    return nm

def noisy_gram_factory(fm, p, shots=None, seed=SEED):
    nm = depolarizing_model(p)
    sim = (AerSimulator(noise_model=nm, seed_simulator=seed) if shots
           else AerSimulator(noise_model=nm, method="density_matrix"))
    n = fm.num_qubits
    def gram(A, B=None):
        sym = B is None
        B = A if sym else B
        circs, idx = [], []
        for i, a in enumerate(A):
            for j, b in enumerate(B):
                if sym and j < i:
                    continue
                tc = to_basis(unitary_overlap(fm.assign_parameters(list(a)),
                                              fm.assign_parameters(list(b))))
                tc.measure_all() if shots else tc.save_probabilities()
                circs.append(tc); idx.append((i, j))
        res = sim.run(circs, shots=shots).result() if shots else sim.run(circs).result()
        K = np.zeros((len(A), len(B)))
        for k, (i, j) in enumerate(idx):
            v = (res.get_counts(k).get("0"*n, 0)/shots) if shots \
                else res.data(k)["probabilities"][0]
            K[i, j] = v
            if sym: K[j, i] = v
        return K
    return gram

# --- the subsample, built once and reported -------------------------------
M_NOISE, F_NOISE = 60, 3
sss = StratifiedShuffleSplit(n_splits=1, train_size=M_NOISE, random_state=SEED)
sub_idx = np.sort(next(sss.split(X_all, y))[0])
X_sub, y_sub = X_all[sub_idx], y[sub_idx]
FOLDS_SUB = list(StratifiedKFold(F_NOISE, shuffle=True,
                                 random_state=SEED).split(X_sub, y_sub))
n_pairs = sum(len(a)*(len(a)-1)//2 + len(b)*len(a) for a, b in FOLDS_SUB)
print(f"noise subsample: M={len(y_sub)}, {F_NOISE} folds, "
      f"class balance {np.bincount((y_sub+1)//2)}")
print(f"circuits per sweep point: {n_pairs:,}  "
      f"(full set would be ~{138000:,})")
print()

print(f"{'p':>7}{'mean':>9}{'s.d.':>8}{'mean K':>10}{'sec':>8}")
noise_rows = []
for p in [0.0, 0.005, 0.01, 0.02]:
    t0 = time.time()
    g = noisy_gram_factory(fm_used, p)
    s, d = cv_kernel(g, prep_b, C=1.0, folds=FOLDS_SUB, data=(X_sub, y_sub))
    print(f"{p:>7}{s.mean():>9.3f}{s.std():>8.3f}{d['mean_K']:>10.3f}"
          f"{time.time()-t0:>8.1f}")
    noise_rows.append((p, s.mean(), d["mean_K"]))

lam = [r[2] / noise_rows[0][2] for r in noise_rows]
print("\nmean K relative to p=0:",
      "  ".join(f"{r[0]}: {l:.3f}" for r, l in zip(noise_rows, lam)))
print("The kernel compresses toward 2^-n; whether the CV score follows is the "
      "question this table answers.")

noise subsample: M=60, 3 folds, class balance [30 30]
circuits per sweep point: 4,740  (full set would be ~138,000)

      p     mean    s.d.    mean K     sec


    0.0    0.533   0.118     0.017    19.0


  0.005    0.533   0.085     0.017    30.3


   0.01    0.583   0.062     0.017    30.5


   0.02    0.583   0.047     0.016    31.2

mean K relative to p=0: 0.0: 1.000  0.005: 0.993  0.01: 0.986  0.02: 0.975
The kernel compresses toward 2^-n; whether the CV score follows is the question this table answers.


### Repeats — and a metric that can actually resolve them

Report **AUC as well as accuracy** here, and this is not a stylistic preference. With $M = 60$ and 3 folds, each fold holds out 20 samples, so accuracy can only take values in steps of $1/20$ and the fold mean in steps of $1/60$. A quantity that moves in jumps of $0.017$ cannot resolve a spread of $0.005$. AUC is continuous and does not have that floor.

The first version of this cell reported accuracy only, gave *identical* means and spreads at $p = 0$ and $p = 0.01$, and I nearly wrote that up as noise-invariance. It was quantisation. If two rows of your table match to four decimal places, suspect your metric before you believe your result.

Note also that the repeat seeds are spaced (`SEED + 100*r`), not consecutive. Adjacent seeds in some generators produce correlated streams, which shrinks the measured spread for no physical reason.

In [13]:
# With shots, repeat the WHOLE evaluation -- the spread is the point.
# Same subsample, so these rows ARE comparable to the ones above.
N_REP = 3                                   # raise to 5+ for the final report
print(f"{'p':>7}{'shots':>7}{'acc':>8}{'+-':>9}{'auc':>8}{'+-':>9}{'mean K':>9}")
for p, sh in [(0.0, 2048), (0.01, 2048), (0.01, 256)]:
    accs, aucs, mks = [], [], []
    for r in range(N_REP):
        g = noisy_gram_factory(fm_used, p, shots=sh, seed=SEED + 100*r)
        sa, d = cv_kernel(g, prep_b, C=1.0, folds=FOLDS_SUB, data=(X_sub, y_sub))
        su, _ = cv_kernel(g, prep_b, C=1.0, folds=FOLDS_SUB, data=(X_sub, y_sub),
                          metric="auc")
        accs.append(sa.mean()); aucs.append(su.mean()); mks.append(d["mean_K"])
    print(f"{p:>7}{sh:>7}{np.mean(accs):>8.3f}{np.std(accs):>9.4f}"
          f"{np.mean(aucs):>8.3f}{np.std(aucs):>9.4f}{np.mean(mks):>9.4f}")

      p  shots     acc       +-     auc       +-   mean K


    0.0   2048   0.556   0.0208   0.609   0.0083   0.0170


   0.01   2048   0.622   0.0079   0.629   0.0173   0.0167


   0.01    256   0.606   0.0157   0.629   0.0042   0.0167


**How to read that table.** Compare rows 2 and 3: the noise level is identical and only the shot budget changed, from 2048 down to 256. The mean barely moves; the AUC spread roughly doubles. So on this problem the run-to-run variability is dominated by **sampling**, not by decoherence — which matches what Module 7 found and is the opposite of where most people expect the cost to be.

That is also the actionable finding. If your spread is shot-limited, more shots fix it and a better device does not. Establishing which of the two you are limited by is worth a paragraph in your write-up.

**Caveat this honestly.** Three repeats on 60 samples estimates a standard deviation with enormous uncertainty of its own. The qualitative ordering is trustworthy; the third decimal place is not. Raise `N_REP` to 5 or more, and say in the write-up how many repeats the numbers came from.

---

## 11. Resource accounting

The table that decides whether the method is usable. Put accuracy on the last row, after the reader has seen the price.

In [14]:
M = len(X_all)
tc = to_basis(unitary_overlap(fm_used.assign_parameters(list(np.zeros(N_FEAT))),
                              fm_used.assign_parameters(list(np.ones(N_FEAT)))))
n2 = sum(1 for i in tc.data if i.operation.name in TWO_Q)
per_fold = (M*4//5)*(M*4//5 - 1)//2 + (M//5)*(M*4//5)
total = per_fold * N_FOLDS

print(f"{'quantity':<38}{'value':>16}")
print("-" * 54)
print(f"{'qubits':<38}{fm_used.num_qubits:>16}")
print(f"{'overlap-circuit depth':<38}{tc.depth():>16}")
print(f"{'two-qubit gates per circuit':<38}{n2:>16}")
print(f"{'circuits per CV run':<38}{total:>16,}")
print(f"{'shots per CV run at %d shots' % SHOTS_KERNEL:<38}{total*SHOTS_KERNEL:>16,}")
print(f"{'device time at 1e5 CLOPS (hours)':<38}{total*SHOTS_KERNEL/1e5/3600:>16.2f}")
print(f"{'... and these numbers are REAL':<38}{'(we ran them)':>16}")
print(f"{'classical baseline circuits':<38}{0:>16}")

quantity                                         value
------------------------------------------------------
qubits                                               6
overlap-circuit depth                               34
two-qubit gates per circuit                         20
circuits per CV run                             34,320
shots per CV run at 2048 shots              70,287,360
device time at 1e5 CLOPS (hours)                  0.20
... and these numbers are REAL           (we ran them)
classical baseline circuits                          0


---

## 12. Your results table

Assemble the final table here and paste it into the write-up. Four tables in total: baselines, ablation, noise, resources.

In [15]:
print("=" * 76)
print("FINAL RESULTS  --  all rows on identical folds, seed", SEED)
print("=" * 76)
print(f"{'model':<34}{'mean':>8}{'s.d.':>8}")
print("-" * 76)
for name, sc in [
    ("Majority class", cv_classical(lambda: DummyClassifier(strategy="most_frequent"), prep_b)),
    ("Logistic regression", cv_classical(lambda: LogisticRegression(max_iter=5000), prep_b)),
    ("SVM (RBF, tuned)", cv_classical(lambda: GridSearchCV(SVC(),
        {"C": [0.1, 1, 10], "gamma": ["scale", 0.1, 1]}, cv=3), prep_b)),
    ("QSVM  ZZ linear, c tuned", s_full),
    ("  ablation: no entanglers (Z only)", s_ne),
    ("  ablation: product kernel", s_prod),
]:
    print(f"{name:<34}{sc.mean():>8.3f}{sc.std():>8.3f}")
print("-" * 76)
print(f"qiskit {qiskit.__version__} | seed {SEED} | {N_FOLDS}-fold | "
      f"{N_FEAT} features | bandwidth {BW_BEST}")

FINAL RESULTS  --  all rows on identical folds, seed 0
model                                 mean    s.d.
----------------------------------------------------------------------------


Majority class                       0.500   0.000
Logistic regression                  0.542   0.046
SVM (RBF, tuned)                     0.892   0.042
QSVM  ZZ linear, c tuned             0.700   0.096
  ablation: no entanglers (Z only)   0.792   0.070
  ablation: product kernel           0.708   0.087
----------------------------------------------------------------------------
qiskit 2.5.1 | seed 0 | 5-fold | 6 features | bandwidth 0.5


---

## 13. Write it up

Two pages. Four tables. Six hundred words.

**Question** — the comparison and the falsifier, three or four sentences.

**Data and method** — dataset, folds, model, hyperparameters and how they were chosen, half a page.

**Results** — the four tables, minimally commented.

**Ablation reading** — one paragraph on what was load-bearing.

**Conclusion** — claim, evidence, comparison, limit. Four sentences:

> *On this 240-sample set the ZZ kernel SVM reached A ± B under stratified 5-fold CV, against C ± D for a tuned RBF SVM on identical folds. The ablation showed the entanglers contributed E. We cannot rule out an advantage smaller than one standard deviation.*

**Reproducibility** — versions, seeds, shot counts, runtime, data source.

---

### Before you submit

Restart the kernel and run all. Then check: question and falsifier stated at the top; one fold object used by every table; majority, logistic and RBF baselines present; ablation table present and read; noise sweep with repeats; resource costs stated; every number has a spread; label-shuffle test passes; versions, seeds and runtime printed.

Ten items. Whichever way your result came out, that is what makes it defensible.